# Multimodal SFT: Chart Reasoning

Fine-tune GPT-4.1 on ChartQA image-and-question examples. This notebook covers only ChartQA preparation, leakage checks, guarded training/reuse, deployment inference, and holdout evaluation. `FOUNDRY_RUN_PAID_JOBS=false` and `FOUNDRY_RUN_LIVE_EVALUATION=false` by default.

## Setup

Copy `../.env.template` to `.env`, run `az login`, and install `../../../requirements.lock`. The notebook fetches ChartQA live and writes generated files only to `outputs/`.

In [ ]:
from __future__ import annotations
import hashlib, io, json, math, os, random, sys, time, urllib.request, zipfile
from collections import Counter
from pathlib import Path
from typing import Any
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from IPython.display import display
from dotenv import load_dotenv
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
DEMO_ROOT=Path.cwd().resolve()
if DEMO_ROOT.name=="notebooks": DEMO_ROOT=DEMO_ROOT.parent
TRACK_ROOT=DEMO_ROOT.parent
sys.path.insert(0,str(TRACK_ROOT/"src"))
from multimodal_common import assert_disjoint, image_uri, live_evaluation_enabled, paid_jobs_enabled, project_clients, resolve_or_create_job, sha256, vision_chat, write_jsonl
load_dotenv(DEMO_ROOT/".env")
ROOT=DEMO_ROOT; OUTPUTS=ROOT/"outputs"; OUTPUTS.mkdir(exist_ok=True)
FOUNDRY_RUN_PAID_JOBS=paid_jobs_enabled(); FOUNDRY_RUN_LIVE_EVALUATION=live_evaluation_enabled(); PROJECT_ENDPOINT=os.getenv("FOUNDRY_PROJECT_ENDPOINT","").strip()
print({"demo_root":str(ROOT),"run_paid_jobs":FOUNDRY_RUN_PAID_JOBS,"run_live_evaluation":FOUNDRY_RUN_LIVE_EVALUATION,"project_configured":bool(PROJECT_ENDPOINT)})

## Verify preserved source artifacts

All committed ChartQA training, validation, and holdout JSONL variants are retained byte-for-byte for historical reproducibility.

In [ ]:
manifest_path=ROOT/"data"/"hash-manifest.csv"
if manifest_path.exists():
    hash_manifest=pd.read_csv(manifest_path)
    for row in hash_manifest.itertuples(index=False):
        if sha256(ROOT/row.preserved_path)!=row.preserved_sha256: raise ValueError(row.preserved_path)
    display(hash_manifest[["dataset","preserved_path","split","rows","byte_match"]])
else:
    print("No committed generated JSONL existed for this source demo; live preparation is authoritative.")

## 2. Chart reasoning

Keep ChartQA's official partitions, retain human-authored examples, sample each partition with seed 42, and encode RGB JPEG at quality 95. The recorded v4 scope was 50 train, 20 validation, and 50 holdout examples.

In [ ]:
CHART_SYSTEM="""Analyze the chart and answer with a concise word, number, or short phrase. Avoid explanation unless necessary."""
def prepare_chartqa(train_n=50,validation_n=20,holdout_n=50):
    from datasets import load_dataset
    ds=load_dataset("HuggingFaceM4/ChartQA")
    source={"train":"train","validation":"val","holdout":"test"}; counts={"train":train_n,"validation":validation_n,"holdout":holdout_n}
    files={}; frames={}; ids={}
    for split,name in source.items():
        frame=ds[name].to_pandas(); frame=frame[frame.human_or_machine==0].sample(counts[split],random_state=42).copy()
        frame["answer"]=frame.label.map(lambda x:x[0] if isinstance(x,(list,np.ndarray)) else x); frame["question"]=frame["query"]
        frame["row_id"]=frame.apply(lambda r:hashlib.sha256((str(r.question)+"\0"+str(r.answer)+"\0"+hashlib.sha256(r.image["bytes"]).hexdigest()).encode()).hexdigest(),axis=1)
        ids[split]=set(frame.row_id); rows=[]
        for i,row in enumerate(frame.itertuples(index=False)):
            with Image.open(io.BytesIO(row.image["bytes"])) as im: uri=image_uri(im,None,95)
            rows.append({"messages":[{"role":"system","content":CHART_SYSTEM},{"role":"user","content":[{"type":"text","text":f"Question [{i}]: {row.question}"},{"type":"image_url","image_url":{"url":uri}}]},{"role":"assistant","content":str(row.answer)}]})
        files[split]=write_jsonl(rows,OUTPUTS/"chart-reasoning"/f"chartqa-{split}.jsonl"); frames[split]=frame
    assert_disjoint(ids)
    lineage={"dataset":"HuggingFaceM4/ChartQA","filter":"human_or_machine == 0","sample_random_state":42,"jpeg_quality":95,"splits":{k:{"source_split":source[k],"rows":len(frames[k]),"sha256":sha256(files[k]),"row_ids":sorted(ids[k])} for k in files}}
    (OUTPUTS/"chart-reasoning"/"lineage.json").write_text(json.dumps(lineage,indent=2),encoding="utf-8")
    print({k:{"rows":len(frames[k]),"sha256":sha256(v)} for k,v in files.items()}); return files,frames
chart_files,chart_frames=prepare_chartqa()

### Train or reuse, then evaluate

The job receives `chart_files["train"]` and `chart_files["validation"]` exactly. Reused jobs are monitored to a successful terminal state. Deploy a succeeded model in Foundry and set both deployment names before enabling `FOUNDRY_RUN_LIVE_EVALUATION`; otherwise the notebook uses checked-in representative evidence without remote inference.

In [ ]:
chart_job=None
chart_reuse_id=os.getenv("FOUNDRY_CHART_FINE_TUNING_JOB_ID","").strip()
if FOUNDRY_RUN_PAID_JOBS or chart_reuse_id:
    if not PROJECT_ENDPOINT: raise RuntimeError("Set FOUNDRY_PROJECT_ENDPOINT to submit or monitor a fine-tuning job.")
    with project_clients(PROJECT_ENDPOINT) as (_,chart_client):
        chart_job=resolve_or_create_job(client=chart_client,reuse_id=chart_reuse_id,model="gpt-4.1-2025-04-14",train=chart_files["train"],validation=chart_files["validation"],suffix="multimodal-chartqa",seed=42,training_type="developerTier",run_paid_jobs=FOUNDRY_RUN_PAID_JOBS)
else: print("Paid training and existing-job monitoring are disabled.")
if chart_job:
    with project_clients(PROJECT_ENDPOINT) as (_,chart_client):
        remote_train=chart_client.files.retrieve(chart_job.training_file); remote_validation=chart_client.files.retrieve(chart_job.validation_file)
    expected_train=f"chartqa-train-{sha256(chart_files['train'])[:16]}.jsonl"; expected_validation=f"chartqa-validation-{sha256(chart_files['validation'])[:16]}.jsonl"
    if remote_train.filename!=expected_train or remote_validation.filename!=expected_validation: raise RuntimeError("The monitored job was not trained on this notebook's exact fetched output.")
    training_evidence={"project_endpoint":PROJECT_ENDPOINT,"job_id":chart_job.id,"status":chart_job.status,"model":chart_job.model,"fine_tuned_model":chart_job.fine_tuned_model,"training_file_id":chart_job.training_file,"validation_file_id":chart_job.validation_file,"training_filename":remote_train.filename,"validation_filename":remote_validation.filename,"training_sha256":sha256(chart_files["train"]),"validation_sha256":sha256(chart_files["validation"])}
    (OUTPUTS/"chart-reasoning"/"notebook-training-evidence.json").write_text(json.dumps(training_evidence,indent=2,default=str),encoding="utf-8")
    display(pd.DataFrame([training_evidence]))

def norm_answer(x:str)->str: return " ".join(x.strip().casefold().replace(",","").split())
def evaluate_chart(frame:pd.DataFrame):
    base=os.getenv("FOUNDRY_CHART_BASE_DEPLOYMENT",""); ft=os.getenv("FOUNDRY_CHART_FINE_TUNED_DEPLOYMENT","")
    if not(PROJECT_ENDPOINT and base and ft): raise RuntimeError("Live evaluation requires the project endpoint and both chart deployments.")
    rows=[]
    with project_clients(PROJECT_ENDPOINT) as (_,client):
        for row in frame.itertuples(index=False):
            with Image.open(io.BytesIO(row.image["bytes"])) as im: uri=image_uri(im,None,95)
            for kind,deployment in (("base",base),("fine_tuned",ft)):
                pred=vision_chat(client,deployment,CHART_SYSTEM,row.question,[uri]); rows.append({"row_id":row.row_id,"model_kind":kind,"answer":str(row.answer),"prediction":pred,"correct":norm_answer(pred)==norm_answer(str(row.answer))})
    out=pd.DataFrame(rows); out.to_csv(OUTPUTS/"chart-reasoning"/"holdout-predictions.csv",index=False)
    scores=out.groupby("model_kind").correct.agg(["count","sum","mean"]); base_accuracy=float(scores.loc["base","mean"]); fine_tuned_accuracy=float(scores.loc["fine_tuned","mean"]); delta=fine_tuned_accuracy-base_accuracy
    paired=out.pivot(index="row_id",columns="model_kind",values="correct"); improved_count=int((~paired.base & paired.fine_tuned).sum()); regressed_count=int((paired.base & ~paired.fine_tuned).sum()); discordant=improved_count+regressed_count
    mcnemar_exact_p=min(1.0,2*sum(math.comb(discordant,i) for i in range(min(improved_count,regressed_count)+1))/(2**discordant)) if discordant else 1.0
    metrics={"holdout_rows":len(frame),"base_deployment":base,"fine_tuned_deployment":ft,"base_correct":int(scores.loc["base","sum"]),"fine_tuned_correct":int(scores.loc["fine_tuned","sum"]),"base_accuracy":base_accuracy,"fine_tuned_accuracy":fine_tuned_accuracy,"accuracy_delta":delta,"improved_examples":improved_count,"regressed_examples":regressed_count,"unchanged_examples":int(len(frame)-discordant),"mcnemar_exact_p":mcnemar_exact_p,"verdict":"improvement" if delta>0 else "regression" if delta<0 else "tie","holdout_sha256":sha256(chart_files["holdout"])}
    (OUTPUTS/"chart-reasoning"/"live-evaluation.json").write_text(json.dumps(metrics,indent=2),encoding="utf-8"); display(scores); display(pd.DataFrame([metrics])); return out
if FOUNDRY_RUN_LIVE_EVALUATION:
    chart_evaluation=evaluate_chart(chart_frames["holdout"])
else:
    print("Live evaluation disabled; showing checked-in representative evidence without remote inference.")
    recorded=json.loads((ROOT/"assets"/"recorded-results.json").read_text(encoding="utf-8"))
    display(pd.DataFrame([recorded["chart_reasoning"]])); display(Image.open(ROOT/"assets"/"recorded-regression.png"))

## Conclusion and next step

The checked-in representative run regressed from 90% to 80% on 10 held-out questions. Successful training alone did not establish quality improvement. When live evaluation is enabled, treat `outputs/chart-reasoning/live-evaluation.json` as authoritative for that run; it records full-holdout accuracy, paired transitions, an exact McNemar p-value, and the improvement/regression verdict. Next, continue to `../02-image-classification` for constrained labels, per-class quality, and latency.

> **Variability:** Small-sample accuracy varies with sampling, encoding, model/deployment version, service load, and judge rules. Actual results may vary by model version, data, configuration, region availability, and service conditions.